# Spline Activations Trade Parameters for Compute in KANs

**Paper:** [https://arxiv.org/abs/2404.19756](https://arxiv.org/abs/2404.19756)  
**Authors:** Ziming Liu, Yixuan Wang, Sachin Vaidya, Fabian Ruehle, James Halverson, Marin Soljačić, Thomas Y. Hou, Max Tegmark  
**Repository:** [https://github.com/KindXiaoming/pykan](https://github.com/KindXiaoming/pykan)  
**License:** MIT  

---

*Reproduction generated by Vivory Research — runs on free-tier hardware (Kaggle T4 / Oracle CPU / GitHub Actions).*
*Produced: 2026-05-21 06:54 UTC*


## 1. Setup

Install dependencies from the paper's `requirements.txt`. Some packages may need GPU-specific wheels — adjust for your Colab/Kaggle runtime.

In [ ]:
!pip install --quiet --upgrade pip
!pip install --quiet matplotlib==3.6.2 numpy==1.24.4 scikit_learn==1.1.3 setuptools==65.5.0 sympy==1.11.1 torch==2.2.2 tqdm==4.66.2 pandas==2.0.1 seaborn pyyaml


## 2. Repository

Clone the reference implementation.

In [ ]:
!git clone --depth 1 https://github.com/KindXiaoming/pykan
%cd pykan
!ls -la


## 3. Dataset

Download the dataset. Replace this cell with the dataset-specific loading code from the repository's README or `scripts/download_data.sh`.

In [ ]:
# TODO: Replace with dataset-specific download/load code.
# Check the repo README for instructions — common patterns:
#   bash scripts/download_data.sh
#   python -m src.data.download
#   from datasets import load_dataset; ds = load_dataset("name")
print("Dataset placeholder — fill in from repo README.")


## 4. Configuration

Core hyperparameters. Consider reducing epochs/batch size to fit free-tier GPU limits (Kaggle T4: 16GB VRAM, 30h/week; Colab: variable).

In [ ]:
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Reduced for free-tier — adjust if you have more GPU budget.
CONFIG = {
    "seed": SEED,
    "max_epochs": 1,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "subset_fraction": 0.1,  # use 10% of data for quick reproduction
}
print(json.dumps(CONFIG, indent=2))


## 5+6. Paper-aware evaluation (auto-generated)

The cell below was generated by Vivory's reproduction agent (Opus 4.7) from the paper's abstract, body, repo README, and claimed_metrics. It performs real measurement on a small subset and writes the result to `/kaggle/working/metrics.json` for the runner to ingest.

In [ ]:
# Reproduction: KAN: Kolmogorov-Arnold Networks (arXiv:2404.19756)
# Claim: a small 2-layer KAN solves a 2D Poisson PDE with comparable/better
# accuracy and far fewer parameters than a wider/deeper MLP.
# We train both with an (FD-Laplacian) physics-informed loss on the Poisson
# problem  Lap(u) = f  on [-1,1]^2, true solution u=sin(pi x)sin(pi y),
# then measure real test-grid MSE and real parameter counts.

# pykan is NOT reliably a clean PyPI install (it pins torch==2.2.2 which would
# clobber Kaggle's CUDA torch). Install the package WITHOUT deps from GitHub.
!pip install -q --no-deps git+https://github.com/KindXiaoming/pykan.git

import os, sys, json, math, warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")  # pykan imports matplotlib; avoid display backend issues

OUT = "/kaggle/working/metrics.json"
os.makedirs("/kaggle/working", exist_ok=True)

def dump(d):
    with open(OUT, "w") as f:
        json.dump(d, f)
    print(d)

try:
    import torch
    import torch.nn as nn

    # If repo was cloned in a previous cell, make sure `import kan` can find it.
    for cand in ("pykan", "PyKAN", "EAGLE"):
        if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "kan")):
            sys.path.insert(0, os.path.abspath(cand))
            break

    from kan import KAN  # noqa

    torch.manual_seed(0)
    import numpy as np
    np.random.seed(0)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    PI = math.pi
    H = 0.02  # finite-difference step for the Laplacian

    # ---- Poisson problem data (synthetic, no external dataset needed) -------
    def true_sol(X):
        return torch.sin(PI * X[:, 0:1]) * torch.sin(PI * X[:, 1:2])

    def source_f(X):  # Lap(true_sol) = -2 pi^2 sin(pi x) sin(pi y)
        return -2.0 * (PI ** 2) * torch.sin(PI * X[:, 0:1]) * torch.sin(PI * X[:, 1:2])

    N_INT, N_BC = 400, 160
    lo, hi = -1.0 + 2 * H, 1.0 - 2 * H
    Xint = torch.rand(N_INT, 2) * (hi - lo) + lo            # interior collocation
    # boundary points on the 4 edges of [-1,1]^2
    t = torch.rand(N_BC, 1) * 2.0 - 1.0
    edge = torch.randint(0, 4, (N_BC, 1))
    Xb = torch.zeros(N_BC, 2)
    Xb[:, 0:1] = torch.where(edge == 0, torch.full_like(t, -1.0),
                  torch.where(edge == 1, torch.full_like(t, 1.0), t))
    Xb[:, 1:2] = torch.where(edge == 2, torch.full_like(t, -1.0),
                  torch.where(edge == 3, torch.full_like(t, 1.0), t))
    f_int = source_f(Xint)

    # dense test grid for the reported MSE (forward-only, real measurement)
    g = torch.linspace(-1.0, 1.0, 50)
    gx, gy = torch.meshgrid(g, g, indexing="ij")
    Xtest = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)
    Utest = true_sol(Xtest)

    assert Xint is not None and Xb is not None and Xtest is not None, \
        "collocation/test tensors must be built [N,2]"

    def laplacian_fd(model, X):
        ex = torch.tensor([[H, 0.0]], device=X.device, dtype=X.dtype)
        ey = torch.tensor([[0.0, H]], device=X.device, dtype=X.dtype)
        u0 = model(X)
        lap = (model(X + ex) + model(X - ex) +
               model(X + ey) + model(X - ey) - 4.0 * u0) / (H * H)
        return lap

    def train_pinn(model, dev, steps, lr, bc_w=10.0):
        Xc = Xint.to(dev); Xbb = Xb.to(dev); fv = f_int.to(dev)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        sch = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, steps // 2), gamma=0.3)
        model.train()
        for it in range(steps):
            opt.zero_grad()
            res = laplacian_fd(model, Xc) - fv
            ub = model(Xbb)
            loss = (res ** 2).mean() + bc_w * (ub ** 2).mean()
            loss.backward()
            opt.step(); sch.step()
            if (it + 1) % max(1, steps // 4) == 0:
                print(f"  step {it+1}/{steps} loss={loss.item():.3e}")
        return model

    def eval_mse(model, dev):
        Xt = Xtest.to(dev); Ut = Utest.to(dev).reshape(-1)
        model.eval()
        with torch.no_grad():
            up = model(Xt).reshape(-1)
        return float(((up - Ut) ** 2).mean().item())

    def nparams(model):
        return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

    kan_mse = mlp_mse = None
    kan_p = mlp_p = None

    # ---------------------------- KAN ---------------------------------------
    try:
        kan = None
        for kw in (dict(width=[2, 10, 1], grid=5, k=3, seed=0, device=device,
                        auto_save=False, save_act=False),
                   dict(width=[2, 10, 1], grid=5, k=3, seed=0, device=device),
                   dict(width=[2, 10, 1], grid=5, k=3, seed=0),
                   dict(width=[2, 10, 1], grid=5, k=3)):
            try:
                kan = KAN(**kw); break
            except TypeError:
                continue
        if kan is None:
            raise RuntimeError("could not construct KAN with any known signature")
        try:
            kan.speed()  # disable slow symbolic branch if available
        except Exception:
            pass
        kdev = next(kan.parameters()).device
        print("Training KAN [2,10,1] PINN...")
        train_pinn(kan, kdev, steps=1000, lr=1e-2)
        kan_mse = eval_mse(kan, kdev)
        kan_p = nparams(kan)
        print(f"KAN: mse={kan_mse:.3e} params={kan_p}")
    except Exception as e:
        print(f"KAN run failed: {type(e).__name__}: {e}")

    # ---------------------------- MLP ---------------------------------------
    try:
        class MLP(nn.Module):
            def __init__(self, width=100):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(2, width), nn.Tanh(),
                    nn.Linear(width, width), nn.Tanh(),
                    nn.Linear(width, width), nn.Tanh(),
                    nn.Linear(width, 1),
                )
                for m in self.net:
                    if isinstance(m, nn.Linear):
                        nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)

            def forward(self, x):
                return self.net(x)

        mlp = MLP(width=100).to(device)
        print("Training MLP (4-layer width-100) PINN...")
        train_pinn(mlp, device, steps=2500, lr=1e-3)
        mlp_mse = eval_mse(mlp, device)
        mlp_p = nparams(mlp)
        print(f"MLP: mse={mlp_mse:.3e} params={mlp_p}")
    except Exception as e:
        print(f"MLP run failed: {type(e).__name__}: {e}")

    if kan_mse is None and mlp_mse is None:
        dump({"infrastructure_error": "both KAN and MLP PINN runs failed"})
    else:
        measured = {
            "mse_pde_kan": kan_mse,
            "mse_pde_mlp": mlp_mse,
            "parameter_count_kan": (float(kan_p) if kan_p is not None else None),
            "parameter_count_mlp": (float(mlp_p) if mlp_p is not None else None),
        }
        # coerce non-finite floats so metrics.json stays valid
        measured = {k: (v if (isinstance(v, (int, float)) and math.isfinite(v)) else None)
                    for k, v in measured.items()}
        dump(measured)

except Exception as e:
    dump({"infrastructure_error": f"{type(e).__name__}: {e}"})

## Appendix — Reproduction policy

This notebook runs on **free-tier hardware only**:

- **Kaggle Notebooks** — T4 GPU, 30h/week quota
- **Oracle Cloud** — ARM 4-core CPU, no GPU
- **GitHub Actions** — 2-core CPU, no GPU, 6h timeout
- **Colab** — variable T4/V100, 12h sessions (manual only)

If the full experiment exceeds these limits, reduce `max_epochs` / `subset_fraction` in the config cell and note the delta in the reproduction report.
